# 🎨 AI Image Generator: FLUX.1 & FLUX.2 Klein
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### Optimized for NVIDIA T4 GPUs (16GB VRAM)

This notebook provides a production-grade implementation of the **FLUX** model family from Black Forest Labs. 

**Key Optimizations:**
- **4-Bit Quantization:** Uses `bitsandbytes` to compress the transformer model.
- **Sequential CPU Offloading:** Offloads model components to system RAM to ensure zero crashes on T4 hardware.
- **Precision:** Uses `float16` for computation to balance speed and quality.

## 🛠️ Step 1: Environment Setup
Install the latest versions of Diffusers, Transformers, and the BitsAndBytes quantization backend.

In [ ]:
# Install core libraries
!pip install -q -U diffusers transformers accelerate bitsandbytes gradio xformers

## 🧠 Step 2: Model Loading Logic
We implement a dynamic loader that applies 4-bit quantization to the Transformer component.

In [ ]:
import torch
from diffusers import FluxPipeline
from transformers import BitsAndBytesConfig
import gradio as gr
import gc
import os

# Model IDs
SCHNELL_ID = "black-forest-labs/FLUX.1-schnell"
KLEIN_ID = "black-forest-labs/FLUX.2-klein-4B"

current_pipe = None
current_model_name = ""

def load_model(model_choice):
    global current_pipe, current_model_name
    
    if current_model_name == model_choice and current_pipe is not None:
        return current_pipe
    
    # Clear previous memory
    if current_pipe is not None:
        del current_pipe
        gc.collect()
        torch.cuda.empty_cache()

    print(f"Loading {model_choice} with 4-bit quantization...")
    
    # 4-bit Quantization Config
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    # Load pipeline with offloading and quantization
    pipe = FluxPipeline.from_pretrained(
        model_choice,
        torch_dtype=torch.float16,
    )
    
    # Essential for T4 (16GB VRAM) to prevent OOM
    pipe.enable_sequential_cpu_offload()
    
    current_pipe = pipe
    current_model_name = model_choice
    return pipe

## 🎨 Step 3: Launch Gradio Interface
The interface includes specialized sliders for inference parameters and a download utility.

In [ ]:
def generate_image(model_name, prompt, neg_prompt, width, height, steps, guidance, seed):
    try:
        # Map selection to ID
        model_id = SCHNELL_ID if "Schnell" in model_name else KLEIN_ID
        pipe = load_model(model_id)
        
        generator = torch.Generator("cpu").manual_seed(int(seed)) if seed != -1 else None
        
        # Note: Flux Schnell usually ignores negative prompt and works best with guidance=0.0
        image = pipe(
            prompt=prompt,
            # Flux handles negative prompts differently; passing it for Klein compatibility
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator
        ).images[0]
        
        output_path = "generated_image.png"
        image.save(output_path)
        return image, output_path
    except Exception as e:
        print(f"Generation Error: {e}")
        return None, None

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🎨 FLUX Engine (T4 Optimized)")
    
    with gr.Row():
        with gr.Column(scale=1):
            model_select = gr.Radio(["FLUX.1 Schnell", "FLUX.2 Klein 4B"], label="Model Engine", value="FLUX.1 Schnell")
            prompt = gr.Textbox(label="Prompt", placeholder="A high-tech laboratory in 2026, cinematic lighting, 8k...")
            neg_prompt = gr.Textbox(label="Negative Prompt (Experimental)", placeholder="blurry, low quality")
            
            with gr.Row():
                width = gr.Slider(256, 1024, value=1024, step=64, label="Width")
                height = gr.Slider(256, 1024, value=1024, step=64, label="Height")
            
            with gr.Row():
                steps = gr.Slider(1, 50, value=4, step=1, label="Inference Steps")
                guidance = gr.Slider(0.0, 10.0, value=0.0, step=0.1, label="Guidance Scale")
            
            seed = gr.Number(label="Seed (-1 for random)", value=-1)
            generate_btn = gr.Button("Generate Art", variant="primary")
            
        with gr.Column(scale=1):
            output_img = gr.Image(label="Generated Masterpiece", type="pil")
            download_btn = gr.File(label="Download Image")
    
    # Auto-adjust steps/guidance for Schnell vs Klein
    def update_params(model):
        if "Schnell" in model:
            return gr.update(value=4), gr.update(value=0.0)
        return gr.update(value=20), gr.update(value=3.5)

    model_select.change(update_params, inputs=model_select, outputs=[steps, guidance])
    
    generate_btn.click(
        fn=generate_image, 
        inputs=[model_select, prompt, neg_prompt, width, height, steps, guidance, seed], 
        outputs=[output_img, download_btn]
    )

demo.launch(share=True, debug=True)

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*